# Response Regimes: Within-Document Sequence Analysis

Each kamerstuk is a **sequence** of Q×A pairs. Prior analyses aggregate across documents and lose this structure. This notebook asks:

1. **Autocorrelation**: does the Court's response type at position *k* predict its response at *k+1*? If deflection or alignment clusters within documents, response is document-level, not question-level.
2. **Document-level DEFL saturation**: are there systematically "deflection-heavy" vs. "engagement-heavy" documents, and what predicts them?
3. **Response regime clustering**: treat each document's answer sequence as a profile and cluster documents by their regime.
4. **Sequence dissimilarity (optional)**: optimal-matching distance between document sequences, with hierarchical clustering.
5. **Regime predictors**: does domain, report type, or year predict which regime a document falls into?

In [ ]:
import sys
sys.path.insert(0, '.')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform

from paths import ANNOTATED_Q_CSV, ANNOTATED_A_CSV, METADATA_CSV, DATA_DIR

A_LABELS   = ['FEIT', 'CAU', 'OOR', 'ADV', 'DEFL']
Q_LABELS   = ['FEIT', 'CAU', 'OOR', 'ADV']
RANK       = {'FEIT': 1, 'CAU': 2, 'OOR': 3, 'ADV': 4}

pd.set_option('display.float_format', '{:.3f}'.format)

## 1. Data preparation

In [ ]:
q    = pd.read_csv(ANNOTATED_Q_CSV, sep='\t')
a    = pd.read_csv(ANNOTATED_A_CSV, sep='\t')
meta = pd.read_csv(METADATA_CSV, sep='\t')
report_meta = pd.read_csv(DATA_DIR / 'report_metadata.csv', sep='\t')

df = q.merge(
    a[['src_id', 'vraag_nr', 'llm_label']]
      .rename(columns={'llm_label': 'a_label'}),
    on=['src_id', 'vraag_nr'], how='inner'
).rename(columns={'llm_label': 'q_label'})

# vergaderjaar is already in q; exclude it from meta to avoid _x/_y suffix
df = df.merge(meta[['src_id', 'commissie', 'n_vragen']], on='src_id', how='left')
df = df.merge(report_meta[['rapport_titel', 'category', 'report_type']],
              on='rapport_titel', how='left')

# Sort within documents by question number
df = df.sort_values(['src_id', 'vraag_nr']).reset_index(drop=True)

# Restrict to valid labels
df = df[df['q_label'].isin(Q_LABELS) & df['a_label'].isin(A_LABELS)]

print(f'Pairs: {len(df):,} | Documents: {df["src_id"].nunique()}')
print(f'A label distribution:\n{df["a_label"].value_counts()}')

## 2. Lag-1 autocorrelation of answer labels

In [ ]:
# Create lag-1 pairs within each document
df['a_label_next'] = df.groupby('src_id')['a_label'].shift(-1)
lags = df[df['a_label_next'].notna()][['a_label', 'a_label_next']].copy()

# Transition matrix: P(A_label[k+1] | A_label[k])
trans = pd.crosstab(lags['a_label'], lags['a_label_next'],
                    rownames=['A[k]'], colnames=['A[k+1]'])
trans_pct = trans.div(trans.sum(axis=1), axis=0) * 100
trans_pct = trans_pct.reindex(index=A_LABELS, columns=A_LABELS, fill_value=0)

print('Lag-1 transition matrix (row % — P(A[k+1] | A[k])):')
display(trans_pct.round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Heatmap of transition matrix
sns.heatmap(trans_pct, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, ax=axes[0], cbar_kws={'label': '%'})
axes[0].set_title('Lag-1 Answer Transition Matrix\n(row = A[k], col = A[k+1])')
axes[0].set_xlabel('A[k+1]')
axes[0].set_ylabel('A[k]')

# Diagonal (self-transition = persistence) vs marginal distribution
marginal = df['a_label'].value_counts(normalize=True).reindex(A_LABELS) * 100
diagonal = pd.Series({lbl: trans_pct.loc[lbl, lbl] for lbl in A_LABELS if lbl in trans_pct.index})

x = np.arange(len(diagonal))
width = 0.35
axes[1].bar(x - width/2, marginal.reindex(diagonal.index), width,
            label='Marginal %', color='steelblue', alpha=0.7)
axes[1].bar(x + width/2, diagonal, width,
            label='Self-transition %', color='firebrick', alpha=0.7)
axes[1].set_xticks(x)
axes[1].set_xticklabels(diagonal.index)
axes[1].set_ylabel('%')
axes[1].set_title('Persistence: marginal vs. self-transition\n(self-transition > marginal = clustering)')
axes[1].legend()
axes[1].axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Test: is the diagonal higher than expected under independence (marginal)?
print('Self-transition excess over marginal (pp):')
for lbl in A_LABELS:
    if lbl in trans_pct.index:
        excess = trans_pct.loc[lbl, lbl] - marginal.get(lbl, 0)
        print(f'  {lbl}: self-transition {trans_pct.loc[lbl, lbl]:.1f}%  '
              f'marginal {marginal.get(lbl, 0):.1f}%  excess +{excess:.1f}pp')

# Chi-square test of independence on the transition matrix
chi2, p, dof, _ = stats.chi2_contingency(trans)
print(f'\nChi-square test of independence: χ²={chi2:.1f}, df={dof}, p={p:.2e}')
print('(Reject H0 → answer type at position k is NOT independent of answer type at k+1)')

## 3. Document-level profiles

In [ ]:
# Build document-level feature matrix
doc_a = df.groupby(['src_id', 'a_label']).size().unstack(fill_value=0)
doc_a = doc_a.reindex(columns=A_LABELS, fill_value=0)

doc_q = df.groupby(['src_id', 'q_label']).size().unstack(fill_value=0)
doc_q = doc_q.reindex(columns=Q_LABELS, fill_value=0).add_prefix('q_')

# Row-normalise to get proportions
doc_a_pct = doc_a.div(doc_a.sum(axis=1), axis=0)
doc_q_pct = doc_q.div(doc_q.sum(axis=1), axis=0)

# Merge with document metadata (one row per document)
# Extract vergaderjaar: may be named vergaderjaar or vergaderjaar_x
vj_col = 'vergaderjaar' if 'vergaderjaar' in df.columns else 'vergaderjaar_x'
doc_meta = df.drop_duplicates('src_id')[['src_id', 'commissie', vj_col,
                                         'category', 'report_type', 'n_vragen']].rename(columns={vj_col: 'vergaderjaar'})
doc_profile = doc_meta.set_index('src_id') \
    .join(doc_a_pct.add_prefix('a_')) \
    .join(doc_q_pct)

# Derived summaries
doc_profile['defl_rate']    = doc_profile['a_DEFL']
doc_profile['oor_a_rate']   = doc_profile['a_OOR']
doc_profile['align_rate']   = df.groupby('src_id').apply(
    lambda g: ((g['q_label'] == g['a_label']) & g['a_label'].isin(['FEIT','CAU','OOR','ADV'])).mean()
)

print(f'Document profiles: {len(doc_profile)}')
print(doc_profile[['defl_rate', 'oor_a_rate', 'align_rate']].describe().round(3))

In [ ]:
# Distribution of per-document DEFL rates
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(doc_profile['defl_rate'].dropna(), bins=20, color='#d62728', edgecolor='white', alpha=0.8)
axes[0].axvline(doc_profile['defl_rate'].mean(), color='black', linestyle='--',
                label=f'Mean = {doc_profile["defl_rate"].mean():.2f}')
axes[0].set_xlabel('DEFL rate (proportion of answers deflected)')
axes[0].set_ylabel('Documents')
axes[0].set_title('Per-document DEFL rate distribution')
axes[0].legend()

axes[1].scatter(doc_profile['defl_rate'], doc_profile['oor_a_rate'],
                alpha=0.4, s=20, c='steelblue')
axes[1].set_xlabel('DEFL rate')
axes[1].set_ylabel('OOR answer rate')
axes[1].set_title('DEFL rate vs. OOR answer rate\n(trade-off or independence?)')
r, p = stats.pearsonr(
    doc_profile[['defl_rate','oor_a_rate']].dropna()['defl_rate'],
    doc_profile[['defl_rate','oor_a_rate']].dropna()['oor_a_rate']
)
axes[1].annotate(f'r={r:.2f}, p={p:.3f}', xy=(0.05, 0.92), xycoords='axes fraction', fontsize=9)

plt.tight_layout()
plt.show()

## 4. Response regime clustering

K-means clustering on the document answer-type proportion vector. Each document is characterised by its (FEIT%, CAU%, OOR%, ADV%, DEFL%) share of answers.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Feature matrix: answer proportions only
feat_cols = [f'a_{l}' for l in A_LABELS]
X = doc_profile[feat_cols].dropna()

# Elbow plot: inertia by k
inertias = []
ks = range(2, 9)
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    km.fit(X)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(list(ks), inertias, 'o-', color='steelblue')
ax.set_xlabel('k')
ax.set_ylabel('Inertia')
ax.set_title('Elbow plot — choose k for regime clustering')
plt.tight_layout()
plt.show()

In [ ]:
# Fit with chosen k — adjust if elbow suggests otherwise
K = 4
km = KMeans(n_clusters=K, random_state=42, n_init=20)
labels_km = km.fit_predict(X)
doc_profile.loc[X.index, 'regime'] = labels_km

# Regime centroids
centroids = X.copy()
centroids['regime'] = labels_km
regime_centres = centroids.groupby('regime')[feat_cols].mean().round(3)
regime_centres.columns = A_LABELS

# Name regimes by dominant answer type
regime_names = regime_centres.idxmax(axis=1).map(
    lambda x: {
        'FEIT': 'Factual',
        'OOR':  'Evaluative',
        'ADV':  'Prescriptive',
        'DEFL': 'Deflecting',
        'CAU':  'Analytical'
    }.get(x, x)
)
regime_centres.index = [f'R{i} ({regime_names[i]})' for i in regime_centres.index]
regime_centres['N'] = pd.Series(labels_km).value_counts().sort_index().values

print('Regime centroids (mean answer proportions):')
display(regime_centres)

In [ ]:
# Radar / bar chart of regime profiles
fig, axes = plt.subplots(1, K, figsize=(14, 4), sharey=True)
colors = ['#1f77b4', '#2ca02c', '#d62728', '#ff7f0e', '#9467bd']

for i, ax in enumerate(axes):
    regime_row = X[labels_km == i].mean()
    label_name = regime_centres.index[i]
    ax.bar(A_LABELS, regime_row.values * 100,
           color=colors[:len(A_LABELS)], edgecolor='white')
    ax.set_title(label_name, fontsize=9)
    ax.set_ylim(0, 70)
    ax.set_ylabel('% of answers' if i == 0 else '')
    n = (labels_km == i).sum()
    ax.set_xlabel(f'N={n} docs', fontsize=8)

plt.suptitle('Response regimes: mean answer-type composition', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# PCA scatter coloured by regime
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X)

fig, ax = plt.subplots(figsize=(8, 6))
regime_palette = dict(zip(range(K), ['#1f77b4','#2ca02c','#d62728','#ff7f0e','#9467bd']))

for r in range(K):
    mask = labels_km == r
    ax.scatter(coords[mask, 0], coords[mask, 1],
               label=regime_centres.index[r],
               alpha=0.6, s=25, color=regime_palette[r])

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
ax.set_title('Document response regimes (PCA of answer-type proportions)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(f'PC1 loadings (answer-type contributions):')
for lbl, load in zip(A_LABELS, pca.components_[0]):
    print(f'  {lbl}: {load:+.3f}')

## 5. Sequence dissimilarity (optimal matching)

Treats each document's answer sequence as a string and computes edit distance between all pairs of documents of comparable length. Then clusters by dissimilarity.

Substitution costs: transitions between *similar* labels (e.g. OOR↔ADV) are cheaper than transitions between distant labels (e.g. FEIT↔DEFL).

In [ ]:
# Encode labels as integers for distance computation
LABEL_INT = {'FEIT': 0, 'CAU': 1, 'OOR': 2, 'ADV': 3, 'DEFL': 4}

# Substitution cost matrix — lower = more similar
# Cognitive distance based on rank; DEFL treated as maximally distant
N_LABELS = 5
subst = np.zeros((N_LABELS, N_LABELS))
for i in range(N_LABELS):
    for j in range(N_LABELS):
        if i == j:
            subst[i, j] = 0
        elif i == 4 or j == 4:       # DEFL involved
            subst[i, j] = 2
        else:                         # both substantive: cognitive distance
            subst[i, j] = abs(i - j)

print('Substitution cost matrix (rows/cols = FEIT,CAU,OOR,ADV,DEFL):')
print(pd.DataFrame(subst, index=list(LABEL_INT.keys()), columns=list(LABEL_INT.keys())))

In [ ]:
def om_distance(s1, s2, subst_matrix, indel_cost=1.5):
    """Optimal matching (edit) distance with indel and substitution costs."""
    n, m = len(s1), len(s2)
    dp = np.zeros((n + 1, m + 1))
    dp[:, 0] = np.arange(n + 1) * indel_cost
    dp[0, :] = np.arange(m + 1) * indel_cost
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            dp[i, j] = min(
                dp[i-1, j]   + indel_cost,
                dp[i, j-1]   + indel_cost,
                dp[i-1, j-1] + subst_matrix[s1[i-1], s2[j-1]]
            )
    return dp[n, m]


# Use documents with 5-20 questions for tractable OM computation
docs_om = df.groupby('src_id').filter(lambda g: 5 <= len(g) <= 20)
doc_ids = docs_om['src_id'].unique()
sequences = [
    docs_om[docs_om['src_id'] == did].sort_values('vraag_nr')['a_label']
              .map(LABEL_INT).dropna().astype(int).tolist()
    for did in doc_ids
]

print(f'Documents for OM analysis: {len(doc_ids)} (5–20 questions each)')
print('Computing pairwise OM distances...')

from itertools import combinations
n = len(sequences)
dist_matrix = np.zeros((n, n))
for i, j in combinations(range(n), 2):
    d = om_distance(sequences[i], sequences[j], subst)
    dist_matrix[i, j] = d
    dist_matrix[j, i] = d

print('Done.')

In [ ]:
# Hierarchical clustering on OM distance matrix
linkage_matrix = linkage(squareform(dist_matrix), method='ward')
N_REGIMES = 4
om_labels = fcluster(linkage_matrix, N_REGIMES, criterion='maxclust') - 1

# Attach OM regime to doc_profile
om_regime_map = pd.Series(om_labels, index=doc_ids)
doc_profile['om_regime'] = doc_profile.index.map(om_regime_map)

# Dendrogram (truncated)
fig, ax = plt.subplots(figsize=(12, 4))
dendrogram(linkage_matrix, ax=ax, truncate_mode='lastp', p=30,
           leaf_rotation=90, leaf_font_size=7)
ax.set_title('Hierarchical clustering of document answer sequences (Ward, OM distance)')
ax.set_xlabel('Document cluster')
ax.set_ylabel('Distance')
plt.tight_layout()
plt.show()

# Profile of OM regimes
docs_om_profile = doc_profile.loc[doc_ids].copy()
om_regime_profile = docs_om_profile.groupby('om_regime')[feat_cols].mean() * 100
om_regime_profile.columns = A_LABELS
om_regime_profile['N'] = pd.Series(om_labels).value_counts().sort_index().values
display(om_regime_profile.round(1))

## 6. Regime predictors: domain, report type, year

In [ ]:
# K-means regime: DEFL rate, OOR rate, alignment rate by domain and report type
dp = doc_profile.dropna(subset=['regime']).copy()
dp['regime'] = dp['regime'].astype(int)

# DEFL rate by report_type
print('=== Mean DEFL rate by report type ===')
rt_defl = dp.groupby('report_type')['defl_rate'].agg(['mean', 'median', 'count'])
display(rt_defl.round(3))

groups_rt = [dp[dp['report_type'] == rt]['defl_rate'].dropna()
             for rt in dp['report_type'].dropna().unique()]
h, p = stats.kruskal(*groups_rt)
print(f'Kruskal-Wallis H={h:.2f}, p={p:.4f}\n')

# DEFL rate by domain
print('=== Mean DEFL rate by policy domain (N >= 10 docs) ===')
dom_defl = dp.groupby('category')['defl_rate'].agg(['mean', 'count'])
dom_defl.columns = ['Mean DEFL rate', 'N docs']
display(dom_defl[dom_defl['N docs'] >= 10].sort_values('Mean DEFL rate', ascending=False).round(3))

In [ ]:
# Regime composition by domain (stacked bar)
regime_by_domain = dp.groupby(['category', 'regime']).size().unstack(fill_value=0)
regime_by_domain_pct = regime_by_domain.div(regime_by_domain.sum(axis=1), axis=0) * 100

# Filter to domains with >= 10 documents
regime_by_domain_pct = regime_by_domain_pct[regime_by_domain.sum(axis=1) >= 10]

ax = regime_by_domain_pct.plot(kind='barh', stacked=True, figsize=(10, 5),
                                colormap='tab10')
ax.set_xlabel('% of documents')
ax.set_title('Response regime composition by policy domain')
# Rename legend entries with regime names
handles, labels = ax.get_legend_handles_labels()
new_labels = [f'Regime {int(l)}' for l in labels]
ax.legend(handles, new_labels, loc='lower right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Temporal trend: mean DEFL rate by year
# Normalise vergaderjaar to start year
dp['year'] = dp['vergaderjaar'].astype(str).str.extract(r'(\d{4})')[0].astype(float)

yr_defl = dp[dp['year'] >= 2011].groupby('year')['defl_rate'].agg(['mean', 'count'])
yr_defl = yr_defl[yr_defl['count'] >= 5]  # min 5 docs per year

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(yr_defl.index, yr_defl['mean'] * 100, 'o-', color='#d62728')
ax.fill_between(yr_defl.index,
                (yr_defl['mean'] - yr_defl['mean'].std()) * 100,
                (yr_defl['mean'] + yr_defl['mean'].std()) * 100,
                alpha=0.15, color='#d62728')
ax.set_xlabel('Parliamentary year (start)')
ax.set_ylabel('Mean document DEFL rate (%)')
ax.set_title('DEFL rate over time (document level, years with ≥ 5 docs)')

# Spearman trend test
r_s, p_s = stats.spearmanr(yr_defl.index, yr_defl['mean'])
ax.annotate(f'Spearman r={r_s:.2f}, p={p_s:.3f}', xy=(0.05, 0.88),
            xycoords='axes fraction', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Multinomial logistic: predict regime from domain + report type + doc size
import statsmodels.formula.api as smf

reg_df = dp.dropna(subset=['regime', 'report_type', 'n_vragen']).copy()
reg_df['log_n_vragen'] = np.log1p(reg_df['n_vragen'])
reg_df['is_verantwoording'] = (reg_df['report_type'] == 'verantwoordingsonderzoek').astype(int)

# OLS on DEFL rate — easily interpretable
ols = smf.ols('defl_rate ~ is_verantwoording + log_n_vragen + C(category)',
              data=reg_df).fit()
print(ols.summary2())

## 7. Within-document position effects

Does the position of a question within a document predict the response type? Does deflection increase as the document progresses (scope exhaustion), or decrease (scope clarified early)?

In [ ]:
# Normalised position: 0 = first question, 1 = last
df['pos_norm'] = df.groupby('src_id')['vraag_nr'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else 0.5
)

# DEFL rate by position decile
df['pos_decile'] = pd.cut(df['pos_norm'], bins=10, labels=False)
pos_defl = df.groupby('pos_decile')['a_label'].apply(lambda x: (x == 'DEFL').mean() * 100)
pos_oor  = df.groupby('pos_decile')['a_label'].apply(lambda x: (x == 'OOR').mean() * 100)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(pos_defl.index, pos_defl.values, 'o-', color='#d62728', label='DEFL %')
ax.plot(pos_oor.index, pos_oor.values, 's--', color='steelblue', label='OOR %')
ax.set_xlabel('Position decile within document (0 = early, 9 = late)')
ax.set_ylabel('% of answers')
ax.set_title('Answer type by position within document')
ax.legend()
plt.tight_layout()
plt.show()

r_defl, p_defl = stats.spearmanr(pos_defl.index, pos_defl.values)
r_oor,  p_oor  = stats.spearmanr(pos_oor.index, pos_oor.values)
print(f'Spearman trend DEFL vs. position: r={r_defl:.3f}, p={p_defl:.3f}')
print(f'Spearman trend OOR  vs. position: r={r_oor:.3f},  p={p_oor:.3f}')